In [ ]:

import resource
resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY)) 

In [ ]:
import os  

def rename_fch_to_fchk(directory: str = ".") -> None:
    
    
    directory = os.path.abspath(directory)  
    print(f"Checking directory: {directory}")  

    
    for name in os.listdir(directory):  
        old_path = os.path.join(directory, name)  

        
        if not os.path.isfile(old_path):  
            continue  

        
        if name.endswith(".fch"):  
            
            new_name = name[:-4] + ".fchk"  
            new_path = os.path.join(directory, new_name)  

            
            if os.path.exists(new_path):  
                print(f"Skip: {old_path} -> {new_path} (target already exists)")  
                continue  

            
            os.rename(old_path, new_path)  
            print(f"Renamed: {old_path} -> {new_path}")  

    print("Renaming finished.")  


rename_fch_to_fchk(".")

In [ ]:
import pandas as pd                         
from pathlib import Path                    

def build_scatter_dataframe(scatter_output_file_path: str) -> pd.DataFrame:
    
    
    in_path = Path(scatter_output_file_path).expanduser().resolve()
    
    
    
    df_raw = pd.read_csv(in_path, sep=r"\s+", header=None, comment="#", engine="python")
    
    
    if df_raw.shape[1] < 5:
        raise ValueError(
            "Public message removed for release."
        )
    
    
    col_x = pd.to_numeric(df_raw.iloc[:, 3], errors="coerce")   
    col_y = pd.to_numeric(df_raw.iloc[:, 4], errors="coerce")   
    
    
    df_scatter_output = pd.DataFrame({
        r"\mathrm{sign}(\lambda_2)\,\rho (a.u.)": col_x,        
        "RDG (a.u.)": col_y                                     
    })
    
    
    df_scatter_output = df_scatter_output.dropna().reset_index(drop=True)
    
    
    filename = in_path.stem
    
    
    out_csv = Path.cwd() / f"{filename}_promolecular.csv"
    df_scatter_output.to_csv(out_csv, index=False, encoding="utf-8")
    
    
    return df_scatter_output

In [ ]:
import numpy as np                                 
import matplotlib as mpl                           
import matplotlib.pyplot as plt                    
from matplotlib.colors import LinearSegmentedColormap  
from matplotlib.ticker import MultipleLocator, FormatStrFormatter  

def plot_nci_rdg(
    df_scatter_output: pd.DataFrame,               
    name: str,                                     
    output: bool = True,                           
    figsize: tuple = (2.3, 2.3),                   
    figname: str = None,                           
    dpi: int = 600                                 
):
    
    
    x = df_scatter_output.iloc[:, 0].to_numpy(dtype=float)   
    y = df_scatter_output.iloc[:, 1].to_numpy(dtype=float)   
    
    
    cmin, cmid, cmax = -0.035, -0.0075, 0.02                
    
    cmap = LinearSegmentedColormap.from_list(
        "nci_rdg",
        [(0.0, "blue"), (0.5, "green"), (1.0, "red")]       
    )
    norm = mpl.colors.Normalize(vmin=cmin, vmax=cmax)        
    
    
    fig, ax = plt.subplots(figsize=figsize)                  
    
    
    mpl.rcParams['font.family'] = 'Arial'                    
    
    ax.tick_params(axis='both', which='major', labelsize=7, width=0.75, length=2.5)
    
    
    
    sc = ax.scatter(
        x, y, c=x, cmap=cmap, norm=norm,                     
        s=1, marker='o', linewidths=0, alpha=1.0             
    )
    
    
    
    ax.set_xlim(-0.05, 0.05)                                 
    ax.set_ylim(0.0, 2.0)                                    
    
    
    ax.xaxis.set_major_locator(MultipleLocator(0.01))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45)                                  
        lbl.set_ha('right')                                   
    
    
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    
    
    
    
    ax.set_xlabel(r'$\mathrm{sign}(\lambda_2)\,\rho\ \mathrm{(a.u.)}$', fontsize=7)  
    ax.set_ylabel('RDG (a.u.)', fontsize=7)
    
    
    
    for spine in ax.spines.values():
        spine.set_linewidth(0.75)
    
    
    cb = plt.colorbar(sc, ax=ax, pad=0.02)                   
    cb.set_label(r'$\mathrm{sign}(\lambda_2)\,\rho\ \mathrm{(a.u.)}$', fontsize=7)   
    cb.ax.tick_params(labelsize=7, width=0.75, length=2.0)   
    cb.outline.set_linewidth(0.75)                           
    
    cbticks = np.arange(cmin, cmax + 1e-9, 0.005)
    cb.set_ticks(cbticks)
    cb.ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))
    
    
    
    if figname is None:
        figname = f"{name}_NCI-RDG_promolecular_plot.png"
    
    
    if output:
        fig.savefig(figname, dpi=dpi, bbox_inches="tight")
    
    
    plt.close(fig)

In [ ]:
import subprocess                            
import sys                                   
import os                                    
from pathlib import Path                     
from typing import List, Optional            

def _run_multiwfn_interactive(
    chkfile: Path,
    cmds: List[str],
    multiwfn_cmd: str = "Multiwfn",
    work_dir: Optional[Path] = None,
    extra_newlines: int = 1
) -> subprocess.CompletedProcess:
    
    
    if work_dir is None:
        work_dir = chkfile.parent

    
    
    lines = [str(chkfile.resolve())] + cmds + ([""] * max(0, extra_newlines))
    std_input = "\n".join(lines) + "\n"

    
    proc = subprocess.run(
        [multiwfn_cmd],
        input=std_input,
        text=True,
        cwd=str(work_dir),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False                                
    )
    return proc


def _safe_rename(src: Path, dst: Path) -> None:
    
    
    if not src.exists():
        return
    
    if dst.exists():
        try:
            dst.unlink()
        except Exception:
            pass
    
    src.rename(dst)

def _run_vmd_batch(
    work_dir: Path,
    filename: str,
    vmd_cmd: str = "vmd",
    rdgfill_proc_path: Optional[Path] = None
) -> subprocess.CompletedProcess:
    
    
    work_dir = work_dir.resolve()

    
    if rdgfill_proc_path is None:
        rdgfill_proc_path = (work_dir / "RDGfill_pro_proc.vmd").resolve()
    else:
        rdgfill_proc_path = rdgfill_proc_path.resolve()

    
    out_tga_path = (work_dir / f"{filename}_NCI_promolecualr.tga").resolve()
    out_vmd_path = (work_dir / f"{filename}_NCI_promolecualr.vmd").resolve()
    out_tga = out_tga_path.as_posix()
    out_vmd = out_vmd_path.as_posix()

    
    vmd_script_path = (work_dir / f"{filename}__auto_render.tcl").resolve()

    
    
    
    script_lines = [
        f"source {{{rdgfill_proc_path.as_posix()}}}",   
        f"draw_NCI_promolecular {{{filename}}}",                     
        f"render TachyonInternal {{{out_tga}}}",        
        f"save_state {{{out_vmd}}}",                    
        "quit"                                          
    ]

    
    vmd_script_path.write_text("\n".join(script_lines), encoding="utf-8")
    wrapper_path = (work_dir / f"{filename}__auto_render_wrapper.tcl").resolve()
    wrapper_path.write_text(f'source \"{vmd_script_path.as_posix()}\"\n', encoding="utf-8")

    
    proc = subprocess.run(
        [vmd_cmd, "-dispdev", "text", "-e", str(wrapper_path)],
        cwd=str(work_dir),                    
        stdout=subprocess.PIPE,               
        stderr=subprocess.PIPE,
        text=True,
        check=False
    )

    
    try:
        vmd_script_path.unlink()
    except Exception:
        pass
    try:
        wrapper_path.unlink()
    except Exception:
        pass

    
    if proc.returncode == 0 and not out_tga_path.exists():
        raise RuntimeError("Public message removed for release.")

    return proc

In [ ]:
def process_checkpoints(
    check_point_file_path: str,
    multiwfn_cmd: str = "Multiwfn",
    vmd_cmd: str = "vmd",
    rdgfill_proc_path: Optional[str] = None
) -> None:
    
    
    root = Path(check_point_file_path).expanduser().resolve()

    
    candidates: List[Path] = []
    if root.is_dir():
        
        patterns = ["*.pdb", "*.xyz", "*.mol", "*.mol2"]
        for pat in patterns:
            candidates.extend(root.rglob(pat))
    elif root.is_file():
        
        if root.suffix.lower() in [".pdb", ".xyz", ".mol", ".mol2"]:
            candidates.append(root)
        else:
            print("Public message removed for release.", file=sys.stderr)
            return
    else:
        
        raise FileNotFoundError("Public message removed for release.")

    
    if not candidates:
        print("Public message removed for release.", file=sys.stderr)
        return

    
    for chk in candidates:
        
        work_dir = chk.parent
        filename = chk.stem

        print("Public message removed for release.")
        print("Public message removed for release.")
        print("Public message removed for release.")

        
        
        nci_cmds = ["20", "2", "2", "2", "1", "3", "0", "0", "q"]
        proc1 = _run_multiwfn_interactive(
            chkfile=chk,
            cmds=nci_cmds,
            multiwfn_cmd=multiwfn_cmd,
            work_dir=work_dir,
            extra_newlines=1
        )
        if proc1.returncode != 0:
            print("Public message removed for release.", file=sys.stderr)

        
        
        
        _safe_rename(work_dir / "func1.cub", work_dir / f"{filename}_func_structure_promolecular.cub")
        _safe_rename(work_dir / "func2.cub", work_dir / f"{filename}_func_iso_promolecular.cub")
        _safe_rename(work_dir / "output.txt", work_dir / f"{filename}_scatter_output_promolecular.txt")

        
        
        proc2 = None
        try:
            proc2 = _run_vmd_batch(
                work_dir=work_dir,
                filename=filename,
                vmd_cmd=vmd_cmd,
                rdgfill_proc_path=(Path(rdgfill_proc_path).resolve() if rdgfill_proc_path else None)
            )
        except RuntimeError as exc:
            print("Public message removed for release.", file=sys.stderr)
        if proc2 is not None and proc2.returncode != 0:
            print("Public message removed for release.", file=sys.stderr)

        
        
        df_scatter_output = build_scatter_dataframe(work_dir / f"{filename}_scatter_output_promolecular.txt")
        
        plot_nci_rdg(
            df_scatter_output=df_scatter_output,
            name=filename,
            output=True,
            figsize=(2.3, 2.3),
            figname=None,   
            dpi=600
        )

        print("Public message removed for release.")

    print("Public message removed for release.")

In [ ]:

process_checkpoints(
    check_point_file_path=".",
    multiwfn_cmd="Multiwfn",            
    vmd_cmd="vmd",                      
    rdgfill_proc_path=None              
)